In [1]:
import json
import pyomo.environ as pyo

## Rebalancing inputs

Expansion of the main model in `main_solver.ipynb`: same model plus the buy/sell variables from the rebalancing section of `solver-latex.md`.

In [ ]:
# Portfolio before rebalancing, {ticker: weight} (same shape as "weights" in solver.output.json).
# Tickers not listed hold weight 0. Placeholder: the non-zero weights of data/chosen_initial_portfolio.json.
w_old = {
    "005930.KS": 0.10295428,
    "NVDA": 0.11190578,
    "LOTB.BR": 0.32387866,
    "NEE": 0.06085918,
    "9697.T": 0.00846871,
    "MCK": 0.05886177,
    "LLY": 0.09021307,
    "ELI.BR": 0.03794546,
    "SUNPHARMA.NS": 0.02580091,
    "RYCEY": 0.00006316,
    "RHM.DE": 0.14804856,
    "BKNG": 0.03100046,
}

v = 1_000_000  # total value of the portfolio before rebalancing (EUR)
g = 10         # fixed cost of modifying a stock (EUR)
p = 0.0005     # proportional cost of a transaction
b = 100        # total rebalancing budget (EUR)

target_return = 1.22  # target portfolio return (annual multiplier)

In [3]:
with open("../../data/expected_returns_covariance_matrix.json") as f:
    data = json.load(f)
    tickers = data["tickers"] #list
    exp_ret = data["expected_returns"] #list

with open("../../data/scenario_cumulative_returns.json") as f:
    scenarios = json.load(f) #nested dict

with open("../../data/tech_tickers.json") as f:
    tech_tickers = json.load(f) #list

In [4]:
exp_ret_annual = [r**52 for r in exp_ret]

In [5]:
def solve_cvar_rebalancing(alpha, scenarios, tickers, expected_returns, target_return,
                           w_old, v, g, p, b, K=None, tech_tickers=None, J=None,
                           tolerance=0.0001, time_limit=10, solver_options=None,
                           weight_tolerance=1e-6):
    """
    Minimize Conditional Value-at-Risk (CVaR) of portfolio returns, starting from an
    existing portfolio and limited by a rebalancing budget. Same model as solve_cvar in
    main_solver.ipynb plus the buy/sell variables and the modification budget.

    Args:
        alpha: CVaR confidence level (e.g., 0.95 for 95% CVaR)
        scenarios: Dict of scenarios with cumulative returns, keyed by scenario ID
                   e.g., {'0': {'TICKER': multiplier, ...}, '1': {...}, ...}
        tickers: List of ticker symbols
        expected_returns: List of expected returns (geometric multipliers, e.g., 1.02)
        target_return: Target portfolio expected return (geometric multiplier)
        w_old: Dict {ticker: weight} of the portfolio before rebalancing (missing tickers = 0)
        v: Total value of the portfolio before rebalancing (EUR)
        g: Fixed cost of modifying a stock (EUR)
        p: Proportional cost of a transaction
        b: Total rebalancing budget (EUR)
        K: Maximum number of stocks in portfolio (None = no cardinality constraint)
        tech_tickers: List of tech ticker symbols (subset of tickers)
        J: Maximum share of portfolio in tech stocks (None = no tech share constraint)
        tolerance: Tolerance band around target return (default 0.0001)
        time_limit: Solver time limit in seconds (default 10)
        solver_options: Optional dict of additional solver options
        weight_tolerance: Allowed deviation of the w_old sum from 1

    Returns:
        Tuple: (weights_dict, gamma_value, cvar_objective, solver_result, rebalance)
        rebalance is a dict with the "buys" and "sells" ({ticker: proportion}), the
        "fixed_cost" and the "variable_cost". If the solve is neither optimal nor timed out
        (e.g. infeasible), weights, gamma, cvar and rebalance are None.
    """
    S = len(scenarios)
    n = len(tickers)
    M = 1.0  # Big-M constant

    # Fail loudly on bad input
    unknown = [t for t in w_old if t not in tickers]
    if unknown:
        raise KeyError(f"Tickers in w_old not found in tickers: {unknown}")
    if abs(sum(w_old.values()) - 1) > weight_tolerance:
        raise ValueError(f"w_old must sum to 1, got {sum(w_old.values())}")
    w_old_vec = [w_old.get(tickers[i], 0.0) for i in range(n)]

    # Create model
    model = pyo.ConcreteModel()

    # Decision variables (non-negativity and binary domain enforced via variable domains)
    model.w = pyo.Var(range(n), domain=pyo.NonNegativeReals)  # Portfolio weights
    model.gamma = pyo.Var(domain=pyo.Reals)  # CVaR threshold
    model.x = pyo.Var(range(n), domain=pyo.Binary)  # Stock selection
    model.z = pyo.Var(range(S), domain=pyo.NonNegativeReals)  # Loss overage
    model.w_buy = pyo.Var(range(n), domain=pyo.NonNegativeReals)  # Amount bought
    model.w_sell = pyo.Var(range(n), domain=pyo.NonNegativeReals)  # Amount sold
    model.y_buy = pyo.Var(range(n), domain=pyo.Binary)  # 1 if stock is bought
    model.y_sell = pyo.Var(range(n), domain=pyo.Binary)  # 1 if stock is sold

    # Objective: minimize CVaR
    coeff = 1.0 / ((1 - alpha) * S)
    model.obj = pyo.Objective(
        expr=model.gamma + coeff * sum(model.z[s] for s in range(S)),
        sense=pyo.minimize
    )

    # Budget constraint
    model.budget = pyo.Constraint(
        expr=sum(model.w[i] for i in range(n)) == 1
    )

    # Target return lower bound
    model.target_lower = pyo.Constraint(
        expr=sum(model.w[i] * expected_returns[i] for i in range(n)) >= target_return - tolerance
    )

    # Target return upper bound
    model.target_upper = pyo.Constraint(
        expr=sum(model.w[i] * expected_returns[i] for i in range(n)) <= target_return + tolerance
    )

    # Loss constraint
    model.loss_constraint = pyo.ConstraintList()
    scenario_keys = list(scenarios.keys())
    for s_idx, s_key in enumerate(scenario_keys):
        loss_expr = 1 - sum(model.w[i] * scenarios[s_key][tickers[i]] for i in range(n))
        model.loss_constraint.add(model.z[s_idx] >= loss_expr - model.gamma)

    # Cardinality constraint and Big-M linkage (only if K is specified)
    if K is not None:
        # Cardinality constraint
        model.cardinality = pyo.Constraint(
            expr=sum(model.x[i] for i in range(n)) <= K
        )

        # Big-M linkage
        model.big_m = pyo.ConstraintList()
        for i in range(n):
            model.big_m.add(model.w[i] <= M * model.x[i])

    # Max share in tech stocks (only if J is specified)
    if J is not None:
        tech_idx = [i for i in range(n) if tickers[i] in tech_tickers]
        model.max_tech_share = pyo.Constraint(
            expr=sum(model.w[i] for i in tech_idx) <= J
        )

    # Buy var enforcement
    model.buy_enforcement = pyo.ConstraintList()
    for i in range(n):
        model.buy_enforcement.add(model.y_buy[i] >= model.w[i] - w_old_vec[i])
        model.buy_enforcement.add(model.w_buy[i] >= model.w[i] - w_old_vec[i])

    # Sell var enforcement
    model.sell_enforcement = pyo.ConstraintList()
    for i in range(n):
        model.sell_enforcement.add(model.y_sell[i] >= w_old_vec[i] - model.w[i])
        model.sell_enforcement.add(model.w_sell[i] >= w_old_vec[i] - model.w[i])

    # Modification costs and budget
    model.fixed_mod_costs = pyo.Expression(
        expr=sum(g * (model.y_sell[i] + model.y_buy[i]) for i in range(n))
    )
    model.var_mod_costs = pyo.Expression(
        expr=sum(v * p * (model.w_sell[i] + model.w_buy[i]) for i in range(n))
    )
    model.mod_budget = pyo.Constraint(
        expr=model.fixed_mod_costs + model.var_mod_costs <= b
    )

    # Create solver
    solver = pyo.SolverFactory("highs")
    solver.options['time_limit'] = time_limit

    # Apply user-provided solver options
    if solver_options:
        for key, value in solver_options.items():
            solver.options[key] = value

    # Solve (don't auto-load: an infeasible solve has no solution to load)
    result = solver.solve(model, load_solutions=False)

    # Check for timeout
    if result.solver.termination_condition == pyo.TerminationCondition.maxTimeLimit:
        print(f"WARNING: Solver timeout at target_return={target_return}")

    # No solution to extract (e.g. infeasible)
    if result.solver.termination_condition not in (pyo.TerminationCondition.optimal,
                                                   pyo.TerminationCondition.maxTimeLimit):
        return None, None, None, result, None

    # Load the solution into the model and extract results
    model.solutions.load_from(result)
    weights = {tickers[i]: pyo.value(model.w[i]) for i in range(n)}
    gamma_value = pyo.value(model.gamma)
    cvar_objective = pyo.value(model.obj)
    rebalance = {
        "buys": {tickers[i]: pyo.value(model.w_buy[i]) for i in range(n) if pyo.value(model.y_buy[i]) > 0.5},
        "sells": {tickers[i]: pyo.value(model.w_sell[i]) for i in range(n) if pyo.value(model.y_sell[i]) > 0.5},
        "fixed_cost": pyo.value(model.fixed_mod_costs),
        "variable_cost": pyo.value(model.var_mod_costs),
    }

    return weights, gamma_value, cvar_objective, result, rebalance

In [6]:
alpha = 0.95
K = 30
J = 0.3
tolerance = 0.001
time_limit = 10

weights, gamma, cvar_obj, result, rebalance = solve_cvar_rebalancing(
    alpha=alpha,
    scenarios=scenarios,
    tickers=tickers,
    expected_returns=exp_ret_annual,
    target_return=target_return,
    w_old=w_old,
    v=v, g=g, p=p, b=b,
    K=K,
    tech_tickers=tech_tickers,
    J=J,
    tolerance=tolerance,
    time_limit=time_limit,
)

print(f"Optimization Result: {result.solver.termination_condition}")
if weights is not None:
    print(f"CVaR Objective Value: {cvar_obj:.6f}")
    print(f"Number of stocks selected: {sum(1 for w in weights.values() if w > 1e-6)}")
    print(f"Bought {len(rebalance['buys'])} stocks, sold {len(rebalance['sells'])} stocks")
    print(f"Fixed costs:    {rebalance['fixed_cost']:.2f} EUR")
    print(f"Variable costs: {rebalance['variable_cost']:.2f} EUR")
    print(f"Total costs:    {rebalance['fixed_cost'] + rebalance['variable_cost']:.2f} EUR (budget {b} EUR)")
    print("\nBuys (proportion of portfolio):")
    for ticker, amount in sorted(rebalance["buys"].items(), key=lambda x: x[1], reverse=True):
        print(f"  {ticker}: +{amount:.4f}")
    print("Sells (proportion of portfolio):")
    for ticker, amount in sorted(rebalance["sells"].items(), key=lambda x: x[1], reverse=True):
        print(f"  {ticker}: -{amount:.4f}")

Optimization Result: optimal
CVaR Objective Value: 0.066219
Number of stocks selected: 14
Bought 2 stocks, sold 1 stocks
Fixed costs:    30.00 EUR
Variable costs: 69.12 EUR
Total costs:    99.12 EUR (budget 100 EUR)

Buys (proportion of portfolio):
  INTC: +0.0482
  PEP: +0.0209
Sells (proportion of portfolio):
  NVDA: -0.0691
